# Chapter 3: Tools and the Autonomy Dial

Chapter 2 ended with the model saying something very reasonable: "I
can't tell from what I was given." The abstract of *Better Answers with
Less Compute* never says how the answers get better, and the model had
no way to find out. A person would just open the paper. The model could
only read what we pasted in before it started.

That's the gap this chapter closes. Instead of handing the model
everything up front, we give it a short list of things it's allowed to
*ask for*: "show me the abstract of paper 0", "show me the full text".
Our code carries out the request and hands back the result. Suddenly the
loop from Chapter 1 is back, only now the model is the one choosing each
step.

And the moment a program can take actions, a new question shows up: how
much should it be allowed to do without asking you first?

:::{card} Chapter 3 of 11 · Foundations

**Where we left off:** The model judges papers by meaning, but it can only see what we paste into its prompt, and it can't ask for more.

**What we fix now:** Give the model tools it can ask for, run them in code, feed the results back into the loop, and then decide how much the scout may do on its own.

**By the end, you can:**
- explain what a tool is, from the model's side and from the code's side
- build a scout that chooses its own next step: read an abstract, read the full text, or make a decision
- describe three levels of autonomy, and see why one good run doesn't tell you which level is safe
:::

```mermaid
flowchart TB
    subgraph F["Foundations"]
        direction LR
        LOOP[Agent loop] --> MOD[Model decides] --> TOOL[Tools and autonomy]
    end
    subgraph D["Continuous Development"]
        direction LR
        SCP[Scope and data] --> V1[Build v1] --> EVL[Evals and deploy]
    end
    subgraph C["Continuous Calibration, repeats"]
        direction LR
        RUN[Real use] --> ERR[Error patterns] --> FIX[Fixes] --> MET[New metrics]
    end
    F --> D --> C

    classDef active fill:#fef3c7,stroke:#d97706,stroke-width:3px,color:#78350f
    classDef done fill:#dbeafe,stroke:#93c5fd,stroke-width:1px,color:#1e3a8a
    classDef pending fill:#f3f4f6,stroke:#e5e7eb,stroke-width:1px,color:#9ca3af
    class TOOL active
    class LOOP,MOD done
    class SCP,V1,EVL,RUN,ERR,FIX,MET pending
```

## The fix: let the model ask for things

A **tool** is just a Python function with a short description. The two
halves live in different places:

- The **model** only ever sees the *name* and the *description*, like
  "read_full_text: show the full text of a paper, use it when the
  abstract is not enough". It can't run anything. It can only reply
  "I'd like to use read_full_text on paper 0".
- Your **code** holds the actual function. When the model asks for a
  tool, your code checks the request, runs the function, and pastes the
  result back into the next prompt.

That pasted result is the **observe** step from Chapter 1. So the
loop is back, with the model in the "think" seat:

```mermaid
flowchart LR
    T["Model thinks:<br/>which tool next?"] --> A["Code acts:<br/>runs the tool"]
    A --> O["Result is added<br/>to the transcript"]
    O --> T
    T --> D["Model decides:<br/>recommend or nothing today"]
```

There's a second win hiding in here. The prompt now only needs the
*titles*. The model pulls in the abstract or full text of the papers it
actually cares about, instead of us pasting everything, which is what
makes a day with hundreds of papers workable.

:::{tip} Try it

The cell below defines the two tools, the prompt, and the loop. Read the
printed prompt first: titles only, a list of actions, and a transcript
section that starts empty. Then read the trace of the run on day 5,
the day the Chapter 2 scout got stuck.

As in Chapter 2, the model here is a hand-written **stand-in**
(`mock_model`), so the page runs without an API key. Its replies were
written for this chapter, not recorded from a real model. The Colab
bonus at the end swaps in a real one.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/03-tools-and-autonomy.ipynb)

If the code below doesn't let you type into it, use the badge above to
open this same notebook in Google Colab, where you can edit and run it
freely.
:::

In [ ]:
import json
import re

interest = ("papers about retrieval-augmented generation (RAG), or about "
            "making language models answer from documents they are given")

day_5 = [
    {"title": "Better Answers with Less Compute",
     "abstract": "We introduce a simple method that improves answer quality on two question answering benchmarks while using less compute. Code is available online.",
     "full_text": "Before answering, our method retrieves the five most similar passages from a Wikipedia index and adds them to the prompt. Because the answer comes from the passages, a much smaller model is enough."},
    {"title": "Faster Training for Speech Models",
     "abstract": "We cut speech model training time in half with a new batching scheme. Word error rate stays the same.",
     "full_text": "Audio clips of similar length are grouped into the same batch so less padding is wasted."},
]

# Tools: a function the code runs, plus a description the model sees.
def read_abstract(papers, i):
    return papers[i]["abstract"]

def read_full_text(papers, i):
    return papers[i]["full_text"]

TOOLS = {
    "read_abstract": (read_abstract, "show the abstract of paper i"),
    "read_full_text": (read_full_text, "show the full text of paper i; use it when the abstract is not enough"),
}
FINAL_ACTIONS = {
    "recommend": "recommend paper i to the researcher",
    "nothing_today": "say that no paper matches (use null for paper)",
}

def build_prompt(papers, transcript):
    titles = "\n".join(f"[{i}] {p['title']}" for i, p in enumerate(papers))
    actions = "\n".join(f"- {name}: {desc}" for name, (_, desc) in TOOLS.items())
    actions += "\n" + "\n".join(f"- {name}: {desc}" for name, desc in FINAL_ACTIONS.items())
    steps = "\n".join(transcript) if transcript else "(nothing yet)"
    return f"""You help a researcher decide which new paper to read today.
Their interest: {interest}

Today's papers (titles only):
{titles}

Actions you can take:
{actions}

What you have done so far:
{steps}

Choose ONE next action. Reply with JSON only:
{{"action": "<action name>", "paper": <paper number or null>, "reason": "<one sentence>"}}"""

print(build_prompt(day_5, []))

Everything the model knows about its abilities is in that "Actions you
can take" list. Now the loop that runs it:

In [ ]:
# Hand-written stand-in replies (NOT recorded from a real model).
# One script per day; the stand-in replies with the next line each step.
scripts = {
    "Better Answers with Less Compute": [
        '{"action": "read_abstract", "paper": 0, "reason": "The title mentions answers, so I want to see what the paper does."}',
        '{"action": "read_full_text", "paper": 0, "reason": "The abstract does not say how the answers are improved."}',
        '{"action": "recommend", "paper": 0, "reason": "The method retrieves Wikipedia passages and answers from them, which is RAG."}',
    ],
}

def mock_model(prompt):
    for first_title, script in scripts.items():
        if first_title in prompt:
            step = prompt.count("Result:")          # how many steps are done so far
            return script[min(step, len(script) - 1)]
    return '{"action": "nothing_today", "paper": null, "reason": "No script for these papers."}'

ask_model = mock_model

def parse_action(reply, papers):
    match = re.search(r"\{.*\}", reply, re.DOTALL)
    if not match:
        return {"action": "error", "reason": "No JSON found in the reply."}
    try:
        d = json.loads(match.group())
    except json.JSONDecodeError:
        return {"action": "error", "reason": "The reply was not valid JSON."}
    if d.get("action") not in TOOLS and d.get("action") not in FINAL_ACTIONS:
        return {"action": "error", "reason": f"Unknown action {d.get('action')!r}."}
    if d["action"] != "nothing_today":
        i = d.get("paper")
        if not (isinstance(i, int) and 0 <= i < len(papers)):
            return {"action": "error", "reason": f"Paper {i!r} does not exist."}
    return d

def finish(papers, d):
    if d["action"] == "nothing_today":
        print("Stop -> Nothing relevant today.")
    else:
        print(f"Recommend '{papers[d['paper']]['title']}'")

def run_scout(papers, max_steps=6):
    transcript = []
    for step in range(1, max_steps + 1):
        d = parse_action(ask_model(build_prompt(papers, transcript)), papers)   # think
        if d["action"] in FINAL_ACTIONS:
            print(f"Step {step}: {d['action']} ({d['reason']})")
            finish(papers, d)
            return
        if d["action"] == "error":
            result = f"Error: {d['reason']}"                                       # tell the model what went wrong
        else:
            tool, _ = TOOLS[d["action"]]
            result = tool(papers, d["paper"])                                       # act
        print(f"Step {step}: {d['action']}({d.get('paper')})  <- {d['reason']}")
        print(f"   Result: {result}")
        transcript.append(f"Step {step}: {d['action']}({d.get('paper')})\nResult: {result}")   # observe
    print("Out of steps.")

run_scout(day_5)

Read the trace top to bottom. The model looks at the abstract, notices
it still can't tell, asks for the full text, and only then decides. The
full text says the method retrieves Wikipedia passages before
answering, so this *is* a RAG paper, and the scout recommends it. In
Chapter 2 this same paper was a dead end.

Two details are worth pointing at. Every result goes into `transcript`,
which is pasted into the next prompt: that's how the model "remembers"
what it already did within a run. And `parse_action` is Chapter 2's
check, grown up: an unknown action or a paper that doesn't exist
becomes an error message the model gets to read on its next step,
instead of a crash.

## The autonomy dial

So far the scout only *reads*. Reading can't hurt anything. But what
you actually want from a scout is for it to *do* something with its
decision, like writing a short summary for your blog's reading list.
That's an action with consequences: other people will see it.

So before adding a `publish_summary` action, decide how much the scout
may do on its own. Think of it as a dial with three settings:

| Setting | The scout... | You... | A wrong decision costs |
|---|---|---|---|
| **suggest** (v1) | recommends a paper | decide everything | a few seconds of your time |
| **draft** (v2) | writes a summary and asks to publish it | approve or reject each one | a moment of reading |
| **act** (v3) | publishes on its own | check later, if at all | a wrong post on your blog |

Nothing about the model changes between these settings. What changes is
*where you stand* in the loop, and so how far a mistake can travel
before someone catches it.

:::{tip} Try it

Run the cell below, then change `autonomy` to `"draft"` and run it
again, then try `human_answer = "yes"`, and finally `"act"`. Watch what
happens after the scout makes the same decision each time.
:::

In [ ]:
autonomy = "suggest"     # try "draft", then "act"
human_answer = "no"      # what you'd say when the scout asks for approval

def summarize(paper):
    return paper["full_text"].split(". ")[0] + "."

def publish_summary(paper):
    print(f"   >>> PUBLISHED on your blog: '{paper['title']}': {summarize(paper)}")

def finish(papers, d):
    if d["action"] == "nothing_today":
        print("Stop -> Nothing relevant today.")
        return
    paper = papers[d["paper"]]
    if autonomy == "suggest":
        print(f"Suggestion: read '{paper['title']}'. What happens next is up to you.")
    elif autonomy == "draft":
        print(f"Draft summary: {summarize(paper)}")
        print(f"Publish this? You answered: {human_answer!r}")
        if human_answer == "yes":
            publish_summary(paper)
        else:
            print("   Not published.")
    elif autonomy == "act":
        publish_summary(paper)

run_scout(day_5)

Same run, same decision, three very different endings. At `"suggest"`
the scout's output is a note to you. At `"draft"` it does the writing,
but nothing leaves your hands without a yes. At `"act"` it goes
straight onto your blog.

On day 5 the decision was right, so `"act"` looks great: you got a
correct post and did nothing. That's exactly why it's tempting to turn
the dial up after one good run.

:::{warning} Where it breaks

The dial controls how far a mistake travels. It does nothing to prevent
the mistake. Try day 6 with the dial at `"act"` (same
[Colab link](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/03-tools-and-autonomy.ipynb) as before, if you need it):
:::

:::{tip} Predict before you run it

Day 6 has a paper called *Retrieval Practice Boosts Exam Scores*.
"Retrieval practice" is a study technique for students: recalling facts
from memory instead of rereading notes. It has nothing to do with RAG.
The scout reads the abstract but not the full text. What do you expect
it to publish?
:::

In [ ]:
day_6 = [
    {"title": "Retrieval Practice Boosts Exam Scores",
     "abstract": "Students who practiced retrieving facts from memory scored 14 percent higher on final exams than students who reread their notes. We studied 300 students across four courses.",
     "full_text": "Retrieval practice means recalling information from memory, for example with flashcards, instead of rereading. No computer systems are involved."},
    {"title": "Quantizing LLMs to 3 Bits",
     "abstract": "We compress a language model's weights to 3 bits with little loss in quality. Memory use falls by 80 percent.",
     "full_text": "Weights are grouped and each group gets its own scale factor before rounding."},
]

scripts["Retrieval Practice Boosts Exam Scores"] = [
    '{"action": "read_abstract", "paper": 0, "reason": "The title mentions retrieval."}',
    '{"action": "recommend", "paper": 0, "reason": "The paper is about retrieval, which matches the interest."}',
]

autonomy = "act"
run_scout(day_6)

The scout saw "retrieving" in the abstract, skipped the full text, and
published a summary of an education study on your AI blog. At
`"suggest"` this would have cost you five seconds: you'd read the
title and shrug. At `"act"` it's public.

Here's the uncomfortable part. Day 5 went perfectly and day 6 went
wrong, with the same model, the same code, and the same prompt. Watching
one run, or two, told you nothing about how often the scout gets it
right. So when is it safe to move the dial from "suggest" to "draft",
or from "draft" to "act"? You can't answer that by watching the scout
work. You need a deliberate way to **build** it for one autonomy level,
**measure** how often it's right, and keep **calibrating** it against
real use, before it earns the next level.

That process is what the rest of this course is about. It has two
halves, Continuous Development (build and measure) and Continuous
Calibration (watch real use, find what breaks, fix it, measure again),
and the next chapter starts the first half.

## Bonus (Colab only): a real model with tools

Open this notebook in Colab to let a real model choose the actions:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/03-tools-and-autonomy.ipynb)

Get a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
(no credit card needed), then paste it into `API_KEY` below. Treat it
like a password: fine to paste into your own private Colab copy while
experimenting, never commit it to a public repository or notebook.

In [ ]:
# Colab only: needs a live network call and an API key.
!pip install -q -U google-genai

from google import genai

API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash"

def gemini_model(prompt):
    return client.models.generate_content(model=MODEL, contents=prompt).text

ask_model = gemini_model
autonomy = "suggest"

for name, papers in [("day 5", day_5), ("day 6", day_6)]:
    print(f"=== {name} ===")
    run_scout(papers)
    print()

Look at which tools the real model asks for, and in what order. Does it
read the full text on day 6, or does it fall for "retrieval" the way
the stand-in did? Run it a few times and see whether the answer stays
the same.

## Quick check

Before moving on, answer for yourself (no peeking ahead):

1. What does the model actually "see" of a tool, and what does your code hold? Why does that split matter?

<details>
<summary>Check your reasoning ▸</summary>

A strong answer says the model only sees the tool's name and description in its prompt, and can only *ask* for it by replying with its name. The code holds the real function and decides whether to run it. That split means nothing happens in the world unless your code carries it out, which is also where you can check, refuse, or limit a request.

</details>

2. The Chapter 3 prompt lists only paper titles. Why is that better than pasting every abstract, as in Chapter 2?

<details>
<summary>Check your reasoning ▸</summary>

A good answer mentions scale: a real day can have hundreds of papers, and pasting every abstract and full text won't fit or will be slow and costly. With tools, the model pulls in details only for the papers that look promising, so the prompt stays small.

</details>

3. What would go wrong if `run_scout` didn't append each result to `transcript`?

<details>
<summary>Check your reasoning ▸</summary>

The model would never see the result of its own actions. It would ask for the same abstract again and again, because as far as its prompt shows, nothing has happened yet. The transcript is the observe step: it's how information from one step reaches the next "think".

</details>

4. On day 6, the scout made the same wrong decision at every dial setting. What actually differed between "suggest" and "act"?

<details>
<summary>Check your reasoning ▸</summary>

Only where the mistake ended up. At "suggest" it's a bad suggestion you can ignore in seconds; at "draft" it's a draft you'd reject; at "act" it's a public post. A strong answer notes that autonomy doesn't change how often the scout is wrong, it changes how much each mistake costs and who catches it.

</details>

5. After day 5, would you have set the dial to "act"? What evidence would you want before doing that?

<details>
<summary>Check your reasoning ▸</summary>

A good answer says one run is an anecdote, not evidence. You'd want to see the scout's decisions on many papers where you already know the right answer, and a number for how often it's right, plus some idea of what kinds of mistakes it makes. That's exactly what the next three chapters build: a scope, labeled data, and an evaluation.

</details>

---

**You can now:** give a model tools it can ask for, run them safely in code, feed results back into the loop, and explain what the autonomy dial changes and what it doesn't.

**But one problem remains:** one good run told us nothing about the next one, and we have no way to know when the scout has earned more autonomy. The next chapter starts building on purpose: pick the narrowest useful version of the scout, write down exactly what it should do, and create examples with known right answers to measure it against.

**Next → Chapter 4: Scope and Curate Data**